In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026/Kaggle_Prepared/val/HS/val_c8608ba7.tif
/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026/Kaggle_Prepared/val/HS/val_bbcdad89.tif
/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026/Kaggle_Prepared/val/HS/val_1d63cdc8.tif
/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026/Kaggle_Prepared/val/HS/val_68f339c4.tif
/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026/Kaggle_Prepared/val/HS/val_da53be12.tif
/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026/Kaggle_Prepared/val/HS/val_36c18096.tif
/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026/Kaggle_Prepared/val/HS/val_c988b239.tif
/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026/Kaggle_Prepared/val/HS/val_6b1a72fd.tif
/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026/Kaggl

In [2]:
import os
import re
import cv2
import timm
import torch
import random
import tifffile as tiff
import numpy as np
import pandas as pd
import torch.nn as nn

from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
BASE = Path("/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026/Kaggle_Prepared/train")

records = []

for rgb in sorted((BASE/"RGB").glob("*.png")):
    name = rgb.stem
    label = name.split("_")[0]

    records.append({
        "id": name,
        "label": label,
        "RGB_path": str(rgb),
        "MS_path": str(BASE/"MS"/f"{name}.tif"),
        "HS_path": str(BASE/"HS"/f"{name}.tif")
    })

df = pd.DataFrame(records)

In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
df["fold"] = -1

for fold, (_, val_idx) in enumerate(skf.split(df, df["label"])):
    df.loc[val_idx, "fold"] = fold

In [6]:
def load_rgb(path):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img,(128,128))
    return img.astype("float32")/255.0

In [7]:
HS_START = 10
HS_END = 111   # exclusive → gives 101 bands

def load_hs_clean(path):
    hs = tiff.imread(path).astype("float32")
    hs = hs[:,:,HS_START:HS_END]
    return hs

In [8]:
def compute_indices(ms):
    blue  = ms[:,:,0]
    green = ms[:,:,1]
    red   = ms[:,:,2]
    rededge = ms[:,:,3]
    nir   = ms[:,:,4]

    ndvi = (nir - red) / (nir + red + 1e-6)
    ndre = (nir - rededge) / (nir + rededge + 1e-6)

    return np.stack([ndvi, ndre], axis=-1)

In [9]:
label_map = {"Health":0,"Other":1,"Rust":2}

class MultiModalDataset(Dataset):
    def __init__(self, df, fold, train=True):
        self.train = train
        self.df = df[df.fold!=fold] if train else df[df.fold==fold]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # RGB
        rgb = load_rgb(row.RGB_path)
        rgb = torch.tensor(rgb).permute(2,0,1).float()

        # MS
        ms = tiff.imread(row.MS_path).astype("float32")
        indices = compute_indices(ms)
        ms = np.concatenate([ms, indices], axis=-1)
        ms = torch.tensor(ms).permute(2,0,1).float()

        # HS
        hs = load_hs_clean(row.HS_path)
        hs = torch.tensor(hs).permute(2,0,1).float()

        label = torch.tensor(label_map[row.label])
        return rgb, ms, hs, label

In [10]:
class RGBEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "convnext_tiny",
            pretrained=True,
            num_classes=0
        )

    def forward(self,x):
        return self.backbone(x)

In [11]:
class MSEncoder(nn.Module):
    def __init__(self, in_ch=7):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(in_ch,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(64,768)

    def forward(self,x):
        x = self.net(x).flatten(1)
        return self.fc(x)

In [12]:
class HSEncoder(nn.Module):
    def __init__(self, bands=101):
        super().__init__()

        # Spectral compression per pixel
        self.spectral = nn.Conv1d(bands, 64, kernel_size=1)

        # Spatial CNN
        self.spatial = nn.Sequential(
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(64, 768)

    def forward(self, x):
        # x: [B, Bands, H, W]

        B, C, H, W = x.shape

        # reshape so Conv1d works along spectral dimension
        x = x.view(B, C, -1)          # [B, Bands, H*W]
        x = self.spectral(x)          # [B, 64, H*W]

        # reshape back to spatial
        x = x.view(B, 64, H, W)

        x = self.spatial(x)
        x = x.flatten(1)

        return self.fc(x)

In [13]:
class FusionAttention(nn.Module):
    def __init__(self, dim=768):
        super().__init__()

        self.gate = nn.Sequential(
            nn.Linear(dim*3, dim),
            nn.ReLU(),
            nn.Linear(dim,3),
            nn.Softmax(dim=1)
        )

    def forward(self,r,m,h):
        concat = torch.cat([r,m,h],1)
        weights = self.gate(concat)

        fused = (
            weights[:,0:1]*r +
            weights[:,1:2]*m +
            weights[:,2:3]*h
        )
        return fused

In [14]:
class FusionNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.rgb = RGBEncoder()
        self.ms  = MSEncoder()
        self.hs  = HSEncoder()

        self.fusion = FusionAttention(768)

        self.classifier = nn.Sequential(
            nn.Linear(768,256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256,3)
        )

    def forward(self,rgb,ms,hs):
        r = self.rgb(rgb)
        m = self.ms(ms)
        h = self.hs(hs)

        fused = self.fusion(r,m,h)
        return self.classifier(fused)

In [15]:
EPOCHS = 25
BATCH_SIZE = 8

best_scores = []

for fold in range(5):

    print(f"\n===== FOLD {fold} =====")

    train_ds = MultiModalDataset(df,fold,True)
    val_ds   = MultiModalDataset(df,fold,False)

    train_loader = DataLoader(train_ds,BATCH_SIZE,shuffle=True,num_workers=2)
    val_loader   = DataLoader(val_ds,BATCH_SIZE,shuffle=False,num_workers=2)

    model = FusionNet().to(device)

    optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = GradScaler()

    best_f1 = 0
    patience = 5
    wait = 0

    for epoch in range(EPOCHS):

        model.train()

        for rgb,ms,hs,y in train_loader:

            rgb,ms,hs,y = rgb.to(device),ms.to(device),hs.to(device),y.to(device)

            optimizer.zero_grad()

            with autocast():
                loss = criterion(model(rgb,ms,hs),y)

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer)
            scaler.update()

        scheduler.step()

        # Validation
        model.eval()
        preds,targets = [],[]

        with torch.no_grad():
            for rgb,ms,hs,y in val_loader:
                out = model(rgb.to(device),ms.to(device),hs.to(device))
                preds += out.argmax(1).cpu().tolist()
                targets += y.tolist()

        f1 = f1_score(targets,preds,average="macro")
        print(f"Epoch {epoch+1} | F1={f1:.4f}")

        if f1 > best_f1:
            best_f1 = f1
            wait = 0
            torch.save(model.state_dict(),f"fold{fold}.pth")
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping")
                break

    best_scores.append(best_f1)

print("CV Mean F1:", np.mean(best_scores))


===== FOLD 0 =====


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

/tmp/ipykernel_24/3487629967.py:21: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_24/3487629967.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 | F1=0.5403


/tmp/ipykernel_24/3487629967.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


...

Epoch 5 | F1=0.5535


/tmp/ipykernel_24/3487629967.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 6 | F1=0.4976


/tmp/ipykernel_24/3487629967.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 7 | F1=0.5615
Early stopping
CV Mean F1: 0.6391215756844217


In [16]:
VAL_PATH = "/kaggle/input/competitions/beyond-visible-spectrum-ai-for-agriculture-2026/Kaggle_Prepared/val"

val_ids = sorted([
    f for f in os.listdir(f"{VAL_PATH}/HS")
    if re.fullmatch(r"val_[0-9a-f]{8}\.tif", f)
])

In [17]:
class TestDataset(Dataset):
    def __init__(self, root, ids):
        self.root=root
        self.ids=ids

    def __len__(self):
        return len(self.ids)

    def __getitem__(self,i):
        name=self.ids[i]

        rgb=torch.tensor(
            load_rgb(f"{self.root}/RGB/{name.replace('.tif','.png')}")
        ).permute(2,0,1).float()

        ms=tiff.imread(f"{self.root}/MS/{name}")
        indices=compute_indices(ms)
        ms=np.concatenate([ms,indices],axis=-1)
        ms=torch.tensor(ms).permute(2,0,1).float()

        hs=load_hs_clean(f"{self.root}/HS/{name}")
        hs=torch.tensor(hs).permute(2,0,1).float()

        return name,rgb,ms,hs

In [18]:
loader = DataLoader(TestDataset(VAL_PATH,val_ids),8,False)

models=[]
for fold in range(5):
    m=FusionNet().to(device)
    m.load_state_dict(torch.load(f"fold{fold}.pth"))
    m.eval()
    models.append(m)

inv={0:"Health",1:"Other",2:"Rust"}
rows=[]

with torch.no_grad():
    for names,rgb,ms,hs in loader:

        rgb,ms,hs=rgb.to(device),ms.to(device),hs.to(device)

        logits=0
        for m in models:
            logits += m(rgb,ms,hs)

        logits/=5
        preds=logits.argmax(1).cpu().tolist()

        for n,p in zip(names,preds):
            rows.append((n,inv[p]))

sub=pd.DataFrame(rows,columns=["Id","Category"])
sub.to_csv("submission.csv",index=False)